In [1]:
#| default_exp registry

In [2]:
#| hide
import json, sys, time
from shutil import which
from tempfile import TemporaryDirectory
from fastcore.test import *
from nbdev.showdoc import *

Where a live in-kernel inspector registers itself, and which of them are still running.

A kernel that starts dhrishti's inspector writes a JSON file naming itself, its pid and the base URL it answers on. This module reads that directory. It imports nothing from dhrishti, so a host that never starts an inspector still costs no optional dependency.

In [3]:
#| export
from __future__ import annotations
import os, subprocess
from fastcore.xtras import Path

In [4]:
#| export
# Mirrors `dhrishti.core.PROFILES`; 'data' is always top-level. Checked below where dhrishti is installed.
PROFILES = {
    'minimal':  {'type': 'hidden', 'function': 'group', 'module': 'hidden', 'special': 'hidden'},
    'standard': {'type': 'group',  'function': 'group', 'module': 'group',  'special': 'group'},
    'full':     {'type': 'top',    'function': 'top',   'module': 'top',    'special': 'group'},
}

# Mirrors `dhrishti.core.SORTS`.
SORTS = ('name', 'recent', 'type', 'size')

#: What a kernel says when it cannot import the inspector, as opposed to when the inspector ran and
#: something else went wrong. The second form is numpy's, whose ABI failure never names itself an
#: ImportError until its eighteen lines of advice are over.
IMPORT_FAULTS = ('ModuleNotFoundError', 'ImportError', 'No module named')

def import_failure(err):
    "Whether an inspector bootstrap error is a missing import, which restarting the kernel cannot fix."
    return any(s in (err or '') for s in IMPORT_FAULTS)

def reg_dir():
    "Where live inspectors register themselves; `$DHRISHTI_REG_DIR` overrides it."
    from fastcore.xdg import xdg_config_home
    d = Path(os.environ.get('DHRISHTI_REG_DIR') or xdg_config_home()/'dhrishti'/'reg')
    d.mkdir(parents=True, exist_ok=True)
    return d

def alive(pid):
    "Is `pid` running and not a zombie? `os.kill(pid, 0)` succeeds for a defunct process too."
    try:
        if int(pid) <= 0: return False   # `os.kill` broadcasts on 0 and -1 instead of asking about one process
        os.kill(int(pid), 0)
    except ProcessLookupError: return False
    except PermissionError: return True
    except Exception: return False
    try:
        st = subprocess.run(['ps', '-o', 'state=', '-p', str(pid)],
                            capture_output=True, text=True, timeout=2).stdout.strip()
        if st[:1] == 'Z': return False
    except Exception: pass   # no `ps`, or too slow; `os.kill` already said yes
    return True

def active():
    "Live registry entries, pruning any whose process has gone. The same sweep dhrishti does."
    out = []
    for f in sorted(reg_dir().glob('*.json')):
        try: e = f.read_json()
        except Exception: continue
        if alive(e.get('pid', -1)): out.append(e)
        else:
            try: f.delete()
            except OSError: pass
    return out

`PROFILES` and `SORTS` are copies of `dhrishti.core.PROFILES` and `dhrishti.core.SORTS`. A host names a profile when it asks a kernel for its variables, and dhrishti is an optional dependency, so the names are held here instead of imported.

A profile says how each kind of name is shown: `top` for its own row, `group` for a collapsed group, `hidden` for not at all. `data` is not a key. Data values are top-level under every profile.

The copy is checked against dhrishti below wherever dhrishti is installed. Where it is not, that check tests nothing.

In [5]:
PROFILES['minimal'], SORTS

({'type': 'hidden',
  'function': 'group',
  'module': 'hidden',
  'special': 'hidden'},
 ('name', 'recent', 'type', 'size'))

In [6]:
#| hide
test_eq({k for p in PROFILES.values() for k in p}, {'type', 'function', 'module', 'special'})
assert all(v in ('top', 'group', 'hidden') for p in PROFILES.values() for v in p.values())
try: from dhrishti.core import PROFILES as _P, SORTS as _S
except ImportError: _P = _S = None
if _P is not None:
    test_eq(PROFILES, _P)
    test_eq(tuple(SORTS), tuple(_S))

`import_failure` separates a bootstrap that could not import the inspector from one that ran and failed for another reason. A missing import survives a restart, so a host that sees one offers an install instead.

The match is a substring anywhere in the error text, not a prefix. numpy prints eighteen lines of advice about its ABI before naming `ImportError` at all.

`None` and the empty string are not import failures.

In [7]:
import_failure("ModuleNotFoundError: No module named 'dhrishti'"), import_failure('ValueError: bad shape')

(True, False)

In [8]:
#| hide
test_eq(import_failure(None), False)
test_eq(import_failure(''), False)
numpy_abi = ('RuntimeError: module compiled against API version 0x10 but this version of numpy is 0xf\n'
             'Consider using a different interpreter, or reinstalling numpy, or ...\n'
             'ImportError: numpy.core.multiarray failed to import')
assert import_failure(numpy_abi), 'the exception name arrives last, so a prefix test would miss it'

`reg_dir` is the directory those entries live in: `$DHRISHTI_REG_DIR` where it is set, otherwise `dhrishti/reg` under the XDG config home. It creates the directory, so a caller may glob it straight away.

In [9]:
#| hide
tmp = TemporaryDirectory()
os.environ['DHRISHTI_REG_DIR'] = str(Path(tmp.name)/'reg')

In [10]:
reg_dir().name, reg_dir().exists()

('reg', True)

`alive` asks whether a pid is a process that could still answer. A child that has exited is not, and `os.kill(pid, 0)` says it is until its parent waits for it, so `ps` decides that case. Where `ps` is missing or slower than two seconds, the answer from `os.kill` stands.

A pid this process may not signal is alive. `PermissionError` means the process exists.

Zero and negative numbers are never alive. `os.kill` broadcasts on those rather than answering about one process.

In [11]:
gone = subprocess.Popen([sys.executable, '-c', '']); gone.wait()
alive(os.getpid()), alive(gone.pid), alive(-1)

(True, False, False)

In [12]:
#| hide
def _zombie(timeout=5):
    "A child that has exited and has not been waited for, or None where `ps` cannot say so."
    if not which('ps'): return None
    p = subprocess.Popen([sys.executable, '-c', ''])
    t = time.time()
    while time.time() - t < timeout:
        st = subprocess.run(['ps', '-o', 'state=', '-p', str(p.pid)], capture_output=True, text=True).stdout.strip()
        if st[:1] == 'Z': return p
        time.sleep(.05)
    p.wait()
    return None
if (z := _zombie()) is not None:
    test_eq(alive(z.pid), False)
    z.wait()

`active` reads every `*.json` in `reg_dir()` in filename order and returns the entries whose process is still running. An entry whose process has gone is deleted, which is the sweep dhrishti itself does.

A file that does not parse is skipped and never deleted. A kernel writing its entry is not a kernel that has stopped.

In [13]:
reg_dir().joinpath('a-live.json').write_json({'name': 'live', 'pid': os.getpid(), 'base': 'http://127.0.0.1:8010'})
reg_dir().joinpath('b-gone.json').write_json({'name': 'gone', 'pid': gone.pid, 'base': 'http://127.0.0.1:8011'})
[e['name'] for e in active()]

['live']

In [14]:
#| hide
assert not (reg_dir()/'b-gone.json').exists(), 'a dead entry is swept, not merely filtered out'
(reg_dir()/'c-half.json').write_text('{"name": "half"')
(reg_dir()/'d-nopid.json').write_json({'name': 'nopid', 'base': 'http://127.0.0.1:8012'})
test_eq([e['name'] for e in active()], ['live'])
assert (reg_dir()/'c-half.json').exists(), 'an unreadable file is skipped, never deleted'
assert not (reg_dir()/'d-nopid.json').exists(), 'an entry naming no pid names no process'

In [15]:
#| hide
tmp.cleanup()